# Phase 2 — Condition 6c: Optuna CV Optimization of Forced-Utilization Gate

Tunes the forced-utilization fusion head (frozen MAE encoder + asymmetric clinical/biomarker dropout + auxiliary MRI loss + per-modality LayerNorm + learned temperature) to find the best accuracy/utilization trade-off, **rigorously** via cross-validated Optuna.

**Search space:** `aux_weight`, `clin_dropout`, `learning_rate`, `weight_decay`.
**Objective:** maximise 5-fold CV mean accuracy, with a soft penalty if mean gate_mri < 0.20 (so we don't trade away the imaging utilization we gained in 6b).
**Rigor:** each trial = 5-fold stratified CV on the train+val pool. Test set held out, evaluated ONCE at the end on the best config. Image features cached once (frozen encoder) so trials are fast.

Baselines to beat: 3b (gate 0.141 / acc 0.931) and 6b (gate 0.317 / acc 0.793). Goal: keep gate ≥ 0.20 while pushing accuracy above 6b's 0.793, ideally toward 3b.

> Colab Pro. ~50 trials × 5 folds of frozen-encoder head training (fast on cached features). Set Runtime → GPU.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!pip install -q "monai==1.5.0" nibabel torchio imbalanced-learn scikit-learn optuna
import os; os._exit(0)


## 1. Verify env

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import torch, numpy, monai, optuna
print('NumPy',numpy.__version__,'| MONAI',monai.__version__,'| Optuna',optuna.__version__)
from monai.networks.nets import SwinUNETR
assert torch.cuda.is_available(); print('GPU:',torch.cuda.get_device_name(0))


## 2. Config

In [ ]:
import os, json, numpy as np, torch, datetime
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torchio as tio, pandas as pd, optuna
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score, matthews_corrcoef, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from monai.networks.nets import SwinUNETR
device=torch.device('cuda')

CONDITION_NAME='condition6c_optuna_cv_forced_gate'
N_TRIALS=50; N_FOLDS=5; EPOCHS=40; PATIENCE=10
GATE_TARGET=0.20   # soft constraint: penalise if mean gate_mri below this

DRIVE='/content/drive/My Drive/ADNI_NewDS/'; RESULTS=os.path.join(DRIVE,'results')
BRAIN_DIR=os.path.join(RESULTS,'processed_mri_scans_brain')
SPLITS=os.path.join(RESULTS,'patient_id_splits.npz')
CSV=os.path.join(RESULTS,'project_data_cleaned.csv'); BIO=os.path.join(RESULTS,'preprocessed_biomarker_sequences.npy')
PHASE2=os.path.join(DRIVE,'Phase2_Collapse_Study'); SSL_DIR=os.path.join(PHASE2,'ssl_pretrain')
MAE_ENCODER=os.path.join(SSL_DIR,'ssl_mae_encoder.pth')
COND_DIR=os.path.join(PHASE2,'results',CONDITION_NAME); os.makedirs(COND_DIR,exist_ok=True)
LAB_LOG=os.path.join(PHASE2,'phase2_lab_log.jsonl')
IMG_SIZE=(96,96,96); FEATURE_SIZE=48; VIT_DIM=768; NUM_CLASSES=3; SEED=42
torch.manual_seed(SEED); np.random.seed(SEED)
assert os.path.exists(MAE_ENCODER), 'Need ssl_mae_encoder.pth (Condition 4 pretraining).'
print('Optuna study:', N_TRIALS, 'trials x', N_FOLDS, 'folds')


## 3. Data + cache MAE image features (compute encoder ONCE)

In [ ]:
sp=np.load(SPLITS,allow_pickle=True)
pids_tr,pids_va,pids_te=list(sp['pids_train']),list(sp['pids_val']),list(sp['pids_test'])
y_tr,y_va,y_te=list(sp['labels_train']),list(sp['labels_val']),list(sp['labels_test'])
pool_pids=pids_tr+pids_va; pool_y=np.array(y_tr+y_va)
clin_df=pd.read_csv(CSV)
FEATS=['AGE','PTGENDER','PTEDUCAT','APOE4','MMSE','ADAS13','RAVLT_immediate','RAVLT_learning','RAVLT_forgetting','FAQ']
patient_clin={p:torch.tensor(g[FEATS].values,dtype=torch.float32) for p,g in clin_df.groupby('PTID')}
bio_raw=np.load(BIO,allow_pickle=True); bio_obj=bio_raw.item() if (hasattr(bio_raw,'dtype') and bio_raw.dtype==object and bio_raw.ndim==0) else bio_raw
patient_bio={k:torch.tensor(np.asarray(v),dtype=torch.float32) for k,v in bio_obj.items()}
CLIN_DIM=len(FEATS); BIO_DIM=next(iter(patient_bio.values())).shape[-1]

class SwinEncoder(nn.Module):
    def __init__(self):
        super().__init__(); self.swin=SwinUNETR(in_channels=1,out_channels=1,feature_size=FEATURE_SIZE,use_checkpoint=True).swinViT; self.pool=nn.AdaptiveAvgPool3d(1)
    def forward(self,x): return self.pool(self.swin(x)[-1]).flatten(1)
encoder=SwinEncoder(); sd=torch.load(MAE_ENCODER,map_location='cpu')
miss,unexp=encoder.load_state_dict(sd,strict=False); encoder=encoder.to(device).eval()
print(f'MAE encoder loaded ({(len(encoder.state_dict())-len(miss))/len(encoder.state_dict()):.0%} keys)')

etf=tio.Compose([tio.Resize(IMG_SIZE), tio.ZNormalization(masking_method=tio.ZNormalization.mean)])
@torch.no_grad()
def img_feat(pid):
    vol=np.load(os.path.join(BRAIN_DIR,f'{pid}_processed.npy'))
    subj=tio.Subject(mri=tio.ScalarImage(tensor=torch.tensor(vol,dtype=torch.float32).unsqueeze(0)))
    return encoder(etf(subj).mri.tensor.unsqueeze(0).to(device)).cpu().squeeze(0)
print('Caching image features...')
img_cache={p:img_feat(p) for p in tqdm(pool_pids+pids_te)}
print('cached',len(img_cache))


## 4. Forced-utilization model (matches 6b) + dataset

In [ ]:
class LSTMBranch(nn.Module):
    def __init__(s,ind,h=128,o=128):
        super().__init__(); s.l=nn.LSTM(ind,h,2,batch_first=True,dropout=0.2); s.f=nn.Linear(h,o); s.r=nn.ReLU()
    def forward(s,x): _,(h,_)=s.l(x); return s.r(s.f(h[-1]))

class ForcedGate(nn.Module):
    def __init__(s,cd,bd,nc=3,clin_dropout=0.2):
        super().__init__(); s.clin_dropout=clin_dropout
        s.proj=nn.Sequential(nn.Linear(VIT_DIM,256),nn.ReLU(),nn.Linear(256,128))
        s.cb=LSTMBranch(cd); s.bb=LSTMBranch(bd)
        s.norm_img=nn.LayerNorm(128); s.norm_clin=nn.LayerNorm(128); s.norm_bio=nn.LayerNorm(128)
        s.gate_lin=nn.Sequential(nn.Linear(384,128),nn.ReLU(),nn.Linear(128,3))
        s.log_temp=nn.Parameter(torch.zeros(1))
        s.clf=nn.Sequential(nn.LayerNorm(128),nn.Linear(128,64),nn.ReLU(),nn.Dropout(0.5),nn.Linear(64,nc))
        s.aux_mri=nn.Sequential(nn.LayerNorm(128),nn.Linear(128,64),nn.ReLU(),nn.Linear(64,nc))
    def forward(s,img_emb,c,b,rg=False,return_aux=False):
        fi=s.norm_img(s.proj(img_emb)); fc=s.norm_clin(s.cb(c)); fb=s.norm_bio(s.bb(b))
        if s.training and s.clin_dropout>0:
            r=torch.rand(2)
            if r[0]<s.clin_dropout: fc=fc*0
            if r[1]<s.clin_dropout: fb=fb*0
        st=torch.stack([fi,fc,fb],1); logits=s.gate_lin(torch.cat([fi,fc,fb],1))
        temp=torch.exp(s.log_temp).clamp(0.2,5.0); w=F.softmax(logits/temp,dim=1)
        fused=(st*w.unsqueeze(-1)).sum(1); out=s.clf(fused)
        if return_aux: return out, s.aux_mri(fi), w
        return (out,w) if rg else out

class CachedDS(Dataset):
    def __init__(s,pids,labels): s.pids=list(pids); s.y=torch.tensor(np.array(labels),dtype=torch.long)
    def __len__(s): return len(s.pids)
    def __getitem__(s,i):
        p=s.pids[i]; return img_cache[p],patient_clin[p],patient_bio[p],s.y[i]
def coll(b):
    im,c,bi,y=zip(*b); return torch.stack(im),pad_sequence(c,batch_first=True),pad_sequence(bi,batch_first=True),torch.stack(y)
print('model+dataset ready')


## 5. Train + evaluate-fold helpers (returns acc AND gate_mri)

In [ ]:
def train_eval_fold(tr_pids,tr_y,va_pids,va_y,params):
    model=ForcedGate(CLIN_DIM,BIO_DIM,NUM_CLASSES,clin_dropout=params['clin_dropout']).to(device)
    trL=DataLoader(CachedDS(tr_pids,tr_y),batch_size=8,shuffle=True,collate_fn=coll)
    vaL=DataLoader(CachedDS(va_pids,va_y),batch_size=8,shuffle=False,collate_fn=coll)
    cw=compute_class_weight('balanced',classes=np.unique(tr_y),y=tr_y)
    crit=nn.CrossEntropyLoss(weight=torch.tensor(cw,dtype=torch.float).to(device))
    opt=torch.optim.AdamW(model.parameters(),lr=params['lr'],weight_decay=params['wd'])
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)
    aw=params['aux_weight']
    best=-1; bs=None; wait=0
    for ep in range(EPOCHS):
        model.train()
        for im,c,b,y in trL:
            im,c,b,y=im.to(device),c.to(device),b.to(device),y.to(device)
            opt.zero_grad(); mo,ao,_=model(im,c,b,return_aux=True)
            loss=crit(mo,y)+aw*crit(ao,y); loss.backward(); opt.step()
        model.eval(); P=[];T=[]
        with torch.no_grad():
            for im,c,b,y in vaL: P+=model(im.to(device),c.to(device),b.to(device)).argmax(1).cpu().tolist(); T+=y.tolist()
        vm=matthews_corrcoef(T,P)
        if vm>best: best=vm; bs={k:v.cpu().clone() for k,v in model.state_dict().items()}; wait=0
        else:
            wait+=1
            if wait>=PATIENCE: break
        sched.step()
    if bs: model.load_state_dict(bs)
    # eval fold: acc + mean gate_mri on the held-out fold
    model.eval(); P=[];T=[]; gm_=[]
    with torch.no_grad():
        for im,c,b,y in vaL:
            o,w=model(im.to(device),c.to(device),b.to(device),rg=True)
            P+=o.argmax(1).cpu().tolist(); T+=y.tolist(); gm_.append(w.cpu().numpy())
    acc=accuracy_score(T,P); gate_mri=float(np.concatenate(gm_).mean(0)[0])
    return acc, gate_mri
print('fold helper ready')


## 6. Optuna study (CV objective with gate constraint)
Objective = mean CV accuracy − penalty·max(0, GATE_TARGET − mean gate_mri). Maximised.

In [ ]:
skf=StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED)
pool_arr=np.array(pool_pids)
PENALTY=2.0  # strength of the soft gate constraint

def objective(trial):
    params={
        'aux_weight': trial.suggest_float('aux_weight',0.0,0.6),
        'clin_dropout': trial.suggest_float('clin_dropout',0.0,0.4),
        'lr': trial.suggest_float('lr',1e-4,3e-3,log=True),
        'wd': trial.suggest_float('wd',1e-6,1e-3,log=True),
    }
    accs=[]; gates=[]
    for tr_idx,va_idx in skf.split(pool_arr,pool_y):
        a,g=train_eval_fold(pool_arr[tr_idx].tolist(),pool_y[tr_idx],pool_arr[va_idx].tolist(),pool_y[va_idx],params)
        accs.append(a); gates.append(g)
    macc=float(np.mean(accs)); mgate=float(np.mean(gates))
    trial.set_user_attr('mean_acc',macc); trial.set_user_attr('mean_gate_mri',mgate); trial.set_user_attr('acc_std',float(np.std(accs)))
    return macc - PENALTY*max(0.0, GATE_TARGET-mgate)

study=optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print('\nBest trial value:',round(study.best_value,4))
print('Best params:',study.best_params)
bt=study.best_trial
print(f"Best trial CV: acc={bt.user_attrs['mean_acc']:.4f}\u00b1{bt.user_attrs['acc_std']:.4f}  gate_mri={bt.user_attrs['mean_gate_mri']:.4f}")
# save all trials
trials_df=[{'number':t.number,'value':t.value,**t.params,**t.user_attrs} for t in study.trials]
json.dump(trials_df, open(os.path.join(COND_DIR,'optuna_trials.json'),'w'), indent=2)


## 7. Refit best config, evaluate ONCE on held-out test

In [ ]:
from sklearn.model_selection import train_test_split
bp=study.best_params
_tp,_vp,_ty,_vy=train_test_split(pool_arr,pool_y,test_size=0.15,stratify=pool_y,random_state=SEED)
# train final on internal split (no test leakage), then test
final=ForcedGate(CLIN_DIM,BIO_DIM,NUM_CLASSES,clin_dropout=bp['clin_dropout']).to(device)
trL=DataLoader(CachedDS(_tp.tolist(),_ty),batch_size=8,shuffle=True,collate_fn=coll)
vaL=DataLoader(CachedDS(_vp.tolist(),_vy),batch_size=8,shuffle=False,collate_fn=coll)
cw=compute_class_weight('balanced',classes=np.unique(_ty),y=_ty)
crit=nn.CrossEntropyLoss(weight=torch.tensor(cw,dtype=torch.float).to(device))
opt=torch.optim.AdamW(final.parameters(),lr=bp['lr'],weight_decay=bp['wd'])
sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=50)
best=-1;bs=None;wait=0
for ep in range(50):
    final.train()
    for im,c,b,y in trL:
        im,c,b,y=im.to(device),c.to(device),b.to(device),y.to(device)
        opt.zero_grad(); mo,ao,_=final(im,c,b,return_aux=True); (crit(mo,y)+bp['aux_weight']*crit(ao,y)).backward(); opt.step()
    final.eval();P=[];T=[]
    with torch.no_grad():
        for im,c,b,y in vaL: P+=final(im.to(device),c.to(device),b.to(device)).argmax(1).cpu().tolist();T+=y.tolist()
    vm=matthews_corrcoef(T,P)
    if vm>best: best=vm;bs={k:v.cpu().clone() for k,v in final.state_dict().items()};wait=0
    else:
        wait+=1
        if wait>=12: break
    sched.step()
if bs: final.load_state_dict(bs)
torch.save(bs,os.path.join(COND_DIR,'best_model.pth'))

teL=DataLoader(CachedDS(pids_te,np.array(y_te)),batch_size=8,shuffle=False,collate_fn=coll)
def gmean(cm):
    g=[]
    for i in range(cm.shape[0]):
        tp=cm[i,i];fn=cm[i,:].sum()-tp;fp=cm[:,i].sum()-tp;tn=cm.sum()-(tp+fn+fp)
        se=tp/(tp+fn) if tp+fn>0 else 0; sp=tn/(tn+fp) if tn+fp>0 else 0; g.append((se*sp)**0.5)
    return float(np.mean(g))
final.eval(); gates=[]
with torch.no_grad():
    for im,c,b,y in teL: _,w=final(im.to(device),c.to(device),b.to(device),rg=True); gates.append(w.cpu().numpy())
gates=np.concatenate(gates); mg=gates.mean(0)
def ab(z=None):
    P=[];T=[]
    with torch.no_grad():
        for im,c,b,y in teL:
            im,c,b=im.to(device),c.to(device),b.to(device)
            if z=='mri':im=torch.zeros_like(im)
            if z=='clin':c=torch.zeros_like(c)
            if z=='bio':b=torch.zeros_like(b)
            P+=final(im,c,b).argmax(1).cpu().tolist();T+=y.tolist()
    cm=confusion_matrix(T,P,labels=[0,1,2]); return accuracy_score(T,P),matthews_corrcoef(T,P),gmean(cm)
abl={t:dict(zip(['acc','mcc','gmean'],ab(z))) for t,z in [('full',None),('mri','mri'),('clin','clin'),('bio','bio')]}
temp=float(torch.exp(final.log_temp).clamp(0.2,5.0).item())
print('='*50);print('FINAL TEST (best Optuna config):')
print('  params:',bp)
print('  GATE:',{k:round(v,4) for k,v in zip(['mri','clin','bio'],mg)})
print('  ABLATION acc:',{k:round(v['acc'],4) for k,v in abl.items()})
print('  full acc:',round(abl['full']['acc'],4),'| MRI contribution:',round(abl['full']['acc']-abl['mri']['acc'],4))


## 8. Log to Drive

In [ ]:
full=abl['full']; mri0=abl['mri']
entry={'timestamp':datetime.datetime.now(datetime.UTC).isoformat(),'condition':CONDITION_NAME,
  'change':'Optuna CV-tuned forced-utilization gate (aux loss + asym dropout), MAE encoder frozen',
  'best_params':bp,'cv_best':{'mean_acc':bt.user_attrs['mean_acc'],'acc_std':bt.user_attrs['acc_std'],'mean_gate_mri':bt.user_attrs['mean_gate_mri']},
  'diagnostic':{'gate':{'mri':float(mg[0]),'clin':float(mg[1]),'bio':float(mg[2])},'ablation':abl,'learned_temp':temp,'img_sim_std':None},
  'mri_contribution':float(full['acc']-mri0['acc']),
  'baseline_ref':{'cond3b_gate':0.141,'cond3b_acc':0.931,'cond6b_gate':0.317,'cond6b_acc':0.793}}
with open(LAB_LOG,'a') as f: f.write(json.dumps(entry)+'\n')
json.dump(entry,open(os.path.join(COND_DIR,'result.json'),'w'),indent=2)
np.save(os.path.join(COND_DIR,'test_gate_weights.npy'),gates)
print('Saved to',COND_DIR)
print('\nSUMMARY vs baselines:')
print(f"  3b:  gate 0.141 / acc 0.931")
print(f"  6b:  gate 0.317 / acc 0.793")
print(f"  6c:  gate {mg[0]:.3f} / acc {full['acc']:.3f}  (CV: {bt.user_attrs['mean_acc']:.3f}\u00b1{bt.user_attrs['acc_std']:.3f})")
print('\nPaste this back to Claude.')
